In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
performance_features = [
    "return_measurement_comparison_percent",
    "return_lag_1q",
    "return_lag_2q",
    "return_lag_3q",
    "return_lag_4q",
    "return_change_1q",
    "return_trailing_4q",
    "return_mean_4q",
    "return_std_4q",
    "return_min_4q",
    "positive_quarter_share_4q",
    "current_return_minus_peer_median",
    "current_return_peer_percentile",
]

volatility_features = [
    "return_investment_five_year_volatility_comparison_percent",
    "return_investment_ten_year_volatility_comparison_percent_clean",
    "volatility_5y_minus_10y",
    "volatility_5y_missing",
    "volatility_10y_missing",
]

saa_features = [
    "strategic_growth_allocation",
    "allocation_equity",
    "allocation_property",
    "allocation_infrastructure",
    "allocation_cash",
    "allocation_alternatives",
    "allocation_credit",
    "allocation_fixed_income_excluding_credit",
    "saa_missing",
]

hedging_features = [
    "weighted_currency_hedging_ratio",
    "currency_hedging_applicable_allocation",
    "currency_hedging_distinct_ratios",
]

strategy_change_features = [
    "strategic_growth_change_1q",
    "equity_allocation_change_1q",
    "cash_allocation_change_1q",
]

baseline_features = (
    performance_features
    + volatility_features
    + saa_features
    + hedging_features
    + strategy_change_features
)

In [3]:
# imprt the data
DATA_PATH = Path(
    "../data/processed/hist_model_mysuper_features.parquet"
)

hist_model_mysuper_features = pd.read_parquet(
    DATA_PATH
)

hist_model_mysuper_features.shape

(13719, 136)

In [4]:
# ensure that the dat is in datetime format
hist_model_mysuper_features[
    "period_end_date"
] = pd.to_datetime(
    hist_model_mysuper_features[
        "period_end_date"
    ]
)

hist_model_mysuper_features[
    "target_end_date"
] = pd.to_datetime(
    hist_model_mysuper_features[
        "target_end_date"
    ]
)

In [5]:
# check coluns
missing_features = [
    col
    for col in baseline_features
    if col not in hist_model_mysuper_features.columns
]

missing_features

[]

### Create the modelling population

In [6]:
target_col = "future_4q_bottom_quartile"

model_data = (
    hist_model_mysuper_features[
        hist_model_mysuper_features[
            "future_4q_target_available_option"
        ].eq(True)
        &
        hist_model_mysuper_features[
            target_col
        ].notna()
    ]
    .copy()
)

# convert target from boolean to 1/0 system
model_data[target_col] = (
    model_data[target_col]
    .astype("int8")
)

# sort the table
entity_keys = [
    "rse_abn",
    "abn_product_identifier",
    "abn_investment_menu_identifier",
    "abn_investment_option_identifier",
]

model_data = (
    model_data
    .sort_values(
        ["period_end_date"]
        + entity_keys
    )
    .reset_index(drop=True)
)

In [7]:
model_data.shape

(11720, 136)

In [8]:
model_data[target_col].value_counts(
    normalize=True
)

future_4q_bottom_quartile
0    0.741553
1    0.258447
Name: proportion, dtype: float64

### Inspect the temporal distribution

In [9]:
# define peer fields
peer_keys = [
    "rse_abn",
    "abn_investment_option_identifier",
    "period_end_date",
]

option_target_nunique = (
    model_data
    .groupby(peer_keys)[target_col]
    .nunique()
)

option_target_nunique.value_counts()

future_4q_bottom_quartile
1    9896
Name: count, dtype: int64

In [10]:
option_target_nunique.max()

np.int64(1)

In [11]:
option_quarter_target = (
    model_data
    .groupby(
        peer_keys,
        as_index=False
    )
    .agg(
        future_4q_bottom_quartile=(
            target_col,
            "first",
        )
    )
)

# quarterly summary
row_summary = (
    model_data
    .groupby("period_end_date")
    .agg(
        modelling_rows=(
            target_col,
            "size",
        ),
        row_positive_rate=(
            target_col,
            "mean",
        ),
        target_end_min=(
            "target_end_date",
            "min",
        ),
        target_end_max=(
            "target_end_date",
            "max",
        ),
    )
)

option_summary = (
    option_quarter_target
    .groupby("period_end_date")
    .agg(
        unique_options=(
            "abn_investment_option_identifier",
            "size",
        ),
        option_positive_rate=(
            target_col,
            "mean",
        ),
    )
)

quarter_summary = (
    row_summary
    .join(option_summary)
)

quarter_summary

,modelling_rows,row_positive_rate,target_end_min,target_end_max,unique_options,option_positive_rate
period_end_date,,,,,,
2014-12-31,185,0.254054,2015-12-31,2015-12-31,153,0.254902
2015-03-31,185,0.318919,2016-03-31,2016-03-31,153,0.254902
2015-06-30,185,0.329730,2016-06-30,2016-06-30,153,0.254902
2015-09-30,185,0.308108,2016-09-30,2016-09-30,153,0.254902
2015-12-31,185,0.302703,2016-12-31,2016-12-31,153,0.254902
2016-03-31,185,0.259459,2017-03-31,2017-03-31,153,0.254902
2016-06-30,186,0.258065,2017-06-30,2017-06-30,154,0.253247
2016-09-30,187,0.262032,2017-09-30,2017-09-30,155,0.258065
2016-12-31,200,0.260000,2017-12-31,2017-12-31,168,0.250000


The temporal inspection looks good enough to proceed, and it supports the split strategy we chose. A few things are especially important.

First, the target timing is internally consistent. For every quarter, target_end_min == target_end_max, and it is exactly four quarters after period_end_date. For example, 2023-03-31 → 2024-03-31 and 2024-06-30 → 2025-06-30.

Because of that, safely continue to the next part.

### Prepare the final test period

For the first baseline, the last 4 available quarters will be considered as the final untouched test set.

In [12]:
# define the available quarters
available_quarters = (
    model_data[
        "period_end_date"
    ]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

len(available_quarters)

42

In [13]:
TEST_QUARTERS = 4

test_quarters = (
    available_quarters[
        -TEST_QUARTERS:
    ]
)

test_start = test_quarters[0]
test_end = test_quarters[-1]

test_start, test_end

(Timestamp('2024-06-30 00:00:00'), Timestamp('2025-03-31 00:00:00'))

### Define data used for development

In [14]:
# define data used for development based on final test start date cutoff
development_mask = (
    model_data[
        "target_end_date"
    ]
    <
    test_start
)

development_data = (
    model_data[
        development_mask
    ]
    .copy()
)

# define the final test data
final_test_data = (
    model_data[
        model_data[
            "period_end_date"
        ].isin(test_quarters)
    ]
    .copy()
)

In [15]:
development_data.shape, final_test_data.shape

((9003, 136), (1369, 136))

### Identify the label-maturation gap

identify rows whose quarters are between the last development prediction quarter and the first test quarter, which simply haven't matured by the simulated test date.

In [16]:
maturation_gap = (
    model_data[
        (model_data["period_end_date"] < test_start)
        &
        (model_data["target_end_date"] >= test_start)
    ]
    .copy()
)

# inspect result
maturation_gap[
    [
        "period_end_date",
        "target_end_date",
    ]
].drop_duplicates().sort_values(
    "period_end_date"
)

,period_end_date,target_end_date
9003,2023-06-30,2024-06-30
9348,2023-09-30,2024-09-30
9682,2023-12-31,2024-12-31
10017,2024-03-31,2025-03-31


### Build the expanding validation folds

In [17]:
# define number of folds and number of quarters inside each validation
N_VALIDATION_FOLDS = 4
VALIDATION_QUARTERS = 4

development_quarters = (
    development_data[
        "period_end_date"
    ]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

len(development_quarters)

34

In [18]:
required_validation_quarters = (
    N_VALIDATION_FOLDS
    * VALIDATION_QUARTERS
)

# Check that there is enough history
assert (
    len(development_quarters)
    >
    required_validation_quarters
)

In [19]:
# Take the latest 16 development quarters as the validation windows
validation_quarters_all = (
    development_quarters[-required_validation_quarters:]
)

# Create 4 blocks
validation_blocks = []

for i in range(N_VALIDATION_FOLDS):
    start = (i * VALIDATION_QUARTERS)
    end = (start + VALIDATION_QUARTERS)

    validation_blocks.append(
        validation_quarters_all[start:end]
    )

In [20]:
# inspect result
for i, block in enumerate(
    validation_blocks,
    start=1,
):
    print(
        f"Fold {i}:",
        block[0],
        "to",
        block[-1],
    )

Fold 1: 2019-06-30 00:00:00 to 2020-03-31 00:00:00
Fold 2: 2020-06-30 00:00:00 to 2021-03-31 00:00:00
Fold 3: 2021-06-30 00:00:00 to 2022-03-31 00:00:00
Fold 4: 2022-06-30 00:00:00 to 2023-03-31 00:00:00


### apply fold-specific purging

In [21]:
folds = []

for fold_number, val_quarters in enumerate(
    validation_blocks,
    start=1,
):

    validation_start = (val_quarters[0])

    validation_end = (val_quarters[-1])

    train_mask = (
        (development_data["period_end_date"] < validation_start)
        &
        (development_data["target_end_date"] < validation_start)
    )

    validation_mask = (
        development_data["period_end_date"].isin(
            val_quarters
        )
    )

    train_data = (
        development_data[train_mask].copy()
    )

    validation_data = (
        development_data[validation_mask].copy()
    )

    folds.append(
        {
            "fold": fold_number,
            "validation_start": validation_start,
            "validation_end": validation_end,
            "train_index": train_data.index,
            "validation_index": validation_data.index,
        }
    )

In [22]:
# audit result
fold_summary = []

for fold in folds:

    train_data = (
        development_data.loc[
            fold["train_index"]
        ]
    )

    validation_data = (
        development_data.loc[
            fold["validation_index"]
        ]
    )

    fold_summary.append(
        {
            "fold":
                fold["fold"],

            "train_start":
                train_data[
                    "period_end_date"
                ].min(),

            "train_end":
                train_data[
                    "period_end_date"
                ].max(),

            "train_target_end_max":
                train_data[
                    "target_end_date"
                ].max(),

            "train_rows":
                len(train_data),

            "train_positive_rate":
                train_data[
                    target_col
                ].mean(),

            "validation_start":
                validation_data[
                    "period_end_date"
                ].min(),

            "validation_end":
                validation_data[
                    "period_end_date"
                ].max(),

            "validation_rows":
                len(validation_data),

            "validation_positive_rate":
                validation_data[
                    target_col
                ].mean(),
        }
    )

fold_summary = pd.DataFrame(
    fold_summary
)

fold_summary

,fold,train_start,train_end,train_target_end_max,train_rows,train_positive_rate,validation_start,validation_end,validation_rows,validation_positive_rate
0,1,2014-12-31,2018-03-31,2019-03-31,2722,0.272594,2019-06-30,2020-03-31,1259,0.249404
1,2,2014-12-31,2019-03-31,2020-03-31,3783,0.265662,2020-06-30,2021-03-31,1387,0.268205
2,3,2014-12-31,2020-03-31,2021-03-31,5042,0.261603,2021-06-30,2022-03-31,1254,0.270335
3,4,2014-12-31,2021-03-31,2022-03-31,6429,0.263027,2022-06-30,2023-03-31,1320,0.250000


In [23]:
(
    fold_summary[
        "train_target_end_max"
    ]
    <
    fold_summary[
        "validation_start"
    ]
).all()

np.True_

In [24]:
# inspect dataypes and missingness
feature_audit = pd.DataFrame(
    {
        "dtype": model_data[
            baseline_features
        ].dtypes.astype(str),

        "missing_count": model_data[
            baseline_features
        ].isna().sum(),

        "missing_rate": model_data[
            baseline_features
        ].isna().mean(),
    }
).sort_values(
    "missing_rate",
    ascending=False,
)

feature_audit

,dtype,missing_count,missing_rate
volatility_5y_minus_10y,float64,8725,0.744454
return_investment_ten_year_volatility_comparison_percent_clean,float64,8719,0.743942
return_investment_five_year_volatility_comparison_percent,float64,6016,0.513311
strategic_growth_change_1q,float64,3577,0.305205
cash_allocation_change_1q,float64,3577,0.305205
equity_allocation_change_1q,float64,3577,0.305205
currency_hedging_applicable_allocation,float64,3217,0.274488
allocation_fixed_income_excluding_credit,float64,3217,0.274488
currency_hedging_distinct_ratios,Int64,3217,0.274488
allocation_infrastructure,float64,3217,0.274488


In [25]:
non_numeric_features = [
    col
    for col in baseline_features
    if not pd.api.types.is_numeric_dtype(
        model_data[col]
    )
    and not pd.api.types.is_bool_dtype(
        model_data[col]
    )
]

non_numeric_features

[]

In [26]:
numeric_features = (
    model_data[
        baseline_features
    ]
    .select_dtypes(
        include=["number"]
    )
    .columns
)

infinite_counts = (
    np.isinf(
        model_data[
            numeric_features
        ]
    )
    .sum()
)

infinite_counts[
    infinite_counts > 0
]

Series([], dtype: Int64)

### Setting up Pipeline

In [27]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

In [28]:
# create dummy model to predict from the training-set class prevalence without learning useful relationships.
dummy_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="prior"
            ),
        ),
    ]
)

# create logistics regression model
logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

In [29]:
# create a metric helper function
def calculate_classification_metrics(
    y_true,
    y_pred,
    y_prob,
):
    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob,
            ),

        "average_precision":
            average_precision_score(
                y_true,
                y_prob,
            ),

        "brier_score":
            brier_score_loss(
                y_true,
                y_prob,
            ),
    }

In [30]:
# put 2 models into a dictionary
baseline_models = {
    "dummy_prior":
        dummy_pipeline,

    "logistic_regression":
        logistic_pipeline,
}

### Baseline model training

In [31]:
fold_results = []

for fold in folds:

    train_data = (
        development_data.loc[
            fold["train_index"]
        ]
    )

    validation_data = (
        development_data.loc[
            fold["validation_index"]
        ]
    )

    X_train = train_data[
        baseline_features
    ]

    y_train = train_data[
        target_col
    ]

    X_validation = validation_data[
        baseline_features
    ]

    y_validation = validation_data[
        target_col
    ]

    for model_name, model_pipeline in (
        baseline_models.items()
    ):

        model = clone(
            model_pipeline
        )

        model.fit(
            X_train,
            y_train,
        )

        validation_probability = (
            model.predict_proba(
                X_validation
            )[:, 1]
        )

        validation_prediction = (
            model.predict(
                X_validation
            )
        )

        metrics = (
            calculate_classification_metrics(
                y_true=y_validation,
                y_pred=validation_prediction,
                y_prob=validation_probability,
            )
        )

        fold_results.append(
            {
                "fold":
                    fold["fold"],

                "model":
                    model_name,

                "train_start":
                    train_data[
                        "period_end_date"
                    ].min(),

                "train_end":
                    train_data[
                        "period_end_date"
                    ].max(),

                "validation_start":
                    validation_data[
                        "period_end_date"
                    ].min(),

                "validation_end":
                    validation_data[
                        "period_end_date"
                    ].max(),

                "train_rows":
                    len(train_data),

                "validation_rows":
                    len(validation_data),

                "validation_positive_rate":
                    y_validation.mean(),

                **metrics,
            }
        )

In [32]:
fold_results = pd.DataFrame(
    fold_results
)

fold_results

,fold,model,train_start,train_end,validation_start,validation_end,train_rows,validation_rows,validation_positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,brier_score
0,1,dummy_prior,2014-12-31,2018-03-31,2019-06-30,2020-03-31,2722,1259,0.249404,0.750596,0.500000,0.000000,0.000000,0.000000,0.500000,0.249404,0.187740
1,1,logistic_regression,2014-12-31,2018-03-31,2019-06-30,2020-03-31,2722,1259,0.249404,0.702145,0.572987,0.382239,0.315287,0.345550,0.493813,0.310312,0.254806
2,2,dummy_prior,2014-12-31,2019-03-31,2020-06-30,2021-03-31,3783,1387,0.268205,0.731795,0.500000,0.000000,0.000000,0.000000,0.500000,0.268205,0.196277
3,2,logistic_regression,2014-12-31,2019-03-31,2020-06-30,2021-03-31,3783,1387,0.268205,0.671233,0.518224,0.312500,0.188172,0.234899,0.646782,0.323816,0.249468
4,3,dummy_prior,2014-12-31,2020-03-31,2021-06-30,2022-03-31,5042,1254,0.270335,0.729665,0.500000,0.000000,0.000000,0.000000,0.500000,0.270335,0.197330
5,3,logistic_regression,2014-12-31,2020-03-31,2021-06-30,2022-03-31,5042,1254,0.270335,0.670654,0.480918,0.191667,0.067847,0.100218,0.493567,0.260131,0.251112
6,4,dummy_prior,2014-12-31,2021-03-31,2022-06-30,2023-03-31,6429,1320,0.250000,0.750000,0.500000,0.000000,0.000000,0.000000,0.500000,0.250000,0.187670
7,4,logistic_regression,2014-12-31,2021-03-31,2022-06-30,2023-03-31,6429,1320,0.250000,0.824242,0.683838,0.791667,0.403030,0.534137,0.737438,0.635967,0.145512


### Compare the models

In [33]:
metric_columns = [
    "roc_auc",
    "average_precision",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "brier_score",
]

model_summary = (
    fold_results
    .groupby("model")[
        metric_columns
    ]
    .agg(
        ["mean", "std"]
    )
)

model_summary

roc_auc           average_precision            \
                       mean       std              mean       std   
model                                                               
dummy_prior          0.5000  0.000000          0.259486  0.011333   
logistic_regression  0.5929  0.120388          0.382556  0.171148   

                    balanced_accuracy           precision             recall  \
                                 mean       std      mean      std      mean   
model                                                                          
dummy_prior                  0.500000  0.000000  0.000000  0.00000  0.000000   
logistic_regression          0.563992  0.088393  0.419518  0.26029  0.243584   

                                    f1            accuracy            \
                         std      mean       std      mean       std   
model                                                                  
dummy_prior          0.00000  0.000000  0.000000  0.740514  0.011333   
logistic_regression  0.14665  0.303701  0.183477  0.717068  0.072948   

                    brier_score            
                           mean       std  
model                                      
dummy_prior            0.192254  0.005271  
logistic_regression    0.225225  0.053188

In [34]:
fold_results[
    [
        "fold",
        "model",
        "validation_start",
        "validation_end",
        "roc_auc",
        "average_precision",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
    ]
]

,fold,model,validation_start,validation_end,roc_auc,average_precision,balanced_accuracy,precision,recall,f1
0,1,dummy_prior,2019-06-30,2020-03-31,0.500000,0.249404,0.500000,0.000000,0.000000,0.000000
1,1,logistic_regression,2019-06-30,2020-03-31,0.493813,0.310312,0.572987,0.382239,0.315287,0.345550
2,2,dummy_prior,2020-06-30,2021-03-31,0.500000,0.268205,0.500000,0.000000,0.000000,0.000000
3,2,logistic_regression,2020-06-30,2021-03-31,0.646782,0.323816,0.518224,0.312500,0.188172,0.234899
4,3,dummy_prior,2021-06-30,2022-03-31,0.500000,0.270335,0.500000,0.000000,0.000000,0.000000
5,3,logistic_regression,2021-06-30,2022-03-31,0.493567,0.260131,0.480918,0.191667,0.067847,0.100218
6,4,dummy_prior,2022-06-30,2023-03-31,0.500000,0.250000,0.500000,0.000000,0.000000,0.000000
7,4,logistic_regression,2022-06-30,2023-03-31,0.737438,0.635967,0.683838,0.791667,0.403030,0.534137


From the result, logistic regression model shows a little bit better result than dummy model, but in terms of accuracy, dummy model is better than logistic regression model. But this might be misleading because 75% of the data is negative (False) and its accuracy is around 74%, which means that it might predicts that all of them are negative.

When looking at each fold, it shows that logistic regression is unstable across all of the evaluation metrics.

##### Fold 1

the ROC-AUC = 0.49, which means that it is essentially random ranking (below 50%) but the balanced accuracy = 0.57, which means that it is above the default threshold. That's possible, because ROC-AUC evaluates ranking across all possible thresholds, whereas balanced accuracy only evaluates the particular classifications produced at the default threshold. But the probability distribution should still need to be investigated because the disagreement between 2 metrics are significant.

##### Fold 2
The ROC-AUC is improving to 0.647 but the recall is bad (recall = 0.188). This might mean that the default threshold catches fewer than 1 in 5 positives? need to check on threshold?

##### Fold 3
ROC-AUC, balanced accuracy and recall are all very low. This means that at this fold, the model essentially fails. This fold covers :
- 2021-06-30
- 2021-09-30
- 2021-12-31
- 2022-03-31

This might be because of some unusual pattern/distribution in certain period that are causing problems?

##### Fold 4
This fold is significantly better than previous folds. The model might identify a relatively small subset of observations as risky, but when it does flag one, it is often correct. But the result is significantly different than previous folds. Why?

The initial logistic-regression baseline demonstrated modest overall discrimination across purged expanding-window validation (mean ROC-AUC ≈ 0.59), but performance varied substantially across time (≈0.49–0.74). The strongest performance occurred in the most recent validation block, while two earlier blocks were close to random ranking. This indicates that predictive relationships may be regime-dependent and motivates further investigation of feature availability, feature-family contribution, temporal stability, and nonlinear modelling before evaluating the untouched final test set.

Try checking out of loop prediction to know the overall validation prediction result.

In [35]:
oof_predictions = []

for fold in folds:

    train_data = (
        development_data.loc[
            fold["train_index"]
        ]
    )

    validation_data = (
        development_data.loc[
            fold["validation_index"]
        ]
    )

    X_train = train_data[
        baseline_features
    ]

    y_train = train_data[
        target_col
    ]

    X_validation = validation_data[
        baseline_features
    ]

    y_validation = validation_data[
        target_col
    ]

    for model_name, model_pipeline in (
        baseline_models.items()
    ):

        model = clone(
            model_pipeline
        )

        model.fit(
            X_train,
            y_train,
        )

        validation_probability = (
            model.predict_proba(
                X_validation
            )[:, 1]
        )

        validation_prediction = (
            model.predict(
                X_validation
            )
        )

        for (
            row_index,
            actual,
            probability,
            prediction,
        ) in zip(
            validation_data.index,
            y_validation,
            validation_probability,
            validation_prediction,
        ):

            oof_predictions.append(
                {
                    "row_index":
                        row_index,

                    "fold":
                        fold["fold"],

                    "model":
                        model_name,

                    "period_end_date":
                        validation_data.loc[
                            row_index,
                            "period_end_date",
                        ],

                    "actual":
                        actual,

                    "probability":
                        probability,

                    "prediction":
                        prediction,
                }
            )

oof_predictions = pd.DataFrame(
    oof_predictions
)

In [36]:
oof_predictions

,row_index,fold,model,period_end_date,actual,probability,prediction
0,3783,1,dummy_prior,2019-06-30,0,0.272594,0
1,3784,1,dummy_prior,2019-06-30,0,0.272594,0
2,3785,1,dummy_prior,2019-06-30,0,0.272594,0
3,3786,1,dummy_prior,2019-06-30,0,0.272594,0
4,3787,1,dummy_prior,2019-06-30,0,0.272594,0
...,...,...,...,...,...,...,...
10435,8998,4,logistic_regression,2023-03-31,0,0.249877,0
10436,8999,4,logistic_regression,2023-03-31,0,0.249877,0
10437,9000,4,logistic_regression,2023-03-31,0,0.249877,0
10438,9001,4,logistic_regression,2023-03-31,0,0.433690,0


In [37]:
# check OOF probabilities of logistic regression
# extract the OOF probabilities of logistic regression
logistic_oof = (
    oof_predictions[
        oof_predictions[
            "model"
        ].eq(
            "logistic_regression"
        )
    ]
    .copy()
)

(
    logistic_oof
    .groupby("fold")
    .agg(
        actual_positive_rate=(
            "actual",
            "mean",
        ),
        predicted_positive_rate=(
            "prediction",
            "mean",
        ),
        mean_probability=(
            "probability",
            "mean",
        ),
        probability_std=(
            "probability",
            "std",
        ),
        probability_min=(
            "probability",
            "min",
        ),
        probability_max=(
            "probability",
            "max",
        ),
    )
)

,actual_positive_rate,predicted_positive_rate,mean_probability,probability_std,probability_min,probability_max
fold,,,,,,
1,0.249404,0.205719,0.227615,0.331958,7.729433e-09,0.999871
2,0.268205,0.161500,0.188626,0.260704,2.170748e-05,0.939256
3,0.270335,0.095694,0.180591,0.197420,1.162865e-03,0.968220
4,0.250000,0.127273,0.274254,0.204252,4.128581e-03,0.935384


The main pattern is that the logistic regression is systematically under-flagging positives in Folds 2–4 at the 0.5 threshold, especially Fold 3.

in Fold 3, Actual positives are about 27%, but only 9.6% of rows are classified positive. This is why Fold 3's evaluation metrics are so bad. The model is assigning most observations probabilities below 0.5, so the default threshold is producing very few alerts. But predicted_positive_rate depends on the threshold whereas mean_probability does not. Since mean_probability = 18%, Fold 3 might have 2 separate problems:
The model is both:
- assigning probabilities that are too low overall, and
- producing relatively few scores above 0.5.

That suggests calibration / temporal shift, not just a bad classification threshold.

in Fold 4, the predicted positive rate is significantly lower than actual positive rate, the difference is larger than fold 1 and 2 even though the evaluation metrics are better. It means many observations are probably receiving probabilities somewhere in the middle while a smaller group receives very high probabilities. The average can therefore be around 0.274 even though only 12.7% exceed 0.50. This means that in Fold 4, the model acts conservative, it flags relatively few observations, but the ones it flags are often genuinely positive.

In summary, The logistic model is not merely suffering from a suboptimal 0.5 classification threshold. Its probability calibration and score distribution vary materially across time. Fold 3 substantially underestimates positive risk, while Fold 1 produces unusually extreme probabilities despite near-random discrimination. Fold 4 shows much stronger ranking ability but remains conservative at the default threshold.

Check performance in quarterly manner:

In [39]:
quarterly_oof_metrics = []

for (
    period,
    period_data,
) in (
    logistic_oof
    .groupby(
        "period_end_date"
    )
):

    metrics = (
        calculate_classification_metrics(
            y_true=period_data[
                "actual"
            ],
            y_pred=period_data[
                "prediction"
            ],
            y_prob=period_data[
                "probability"
            ],
        )
    )

    quarterly_oof_metrics.append(
        {
            "period_end_date":
                period,

            "rows":
                len(period_data),

            "positive_rate":
                period_data[
                    "actual"
                ].mean(),

            **metrics,
        }
    )

quarterly_oof_metrics = (
    pd.DataFrame(
        quarterly_oof_metrics
    )
)

quarterly_oof_metrics[
    [
        "period_end_date",
        "rows",
        "positive_rate",
        "roc_auc",
        "average_precision",
        "balanced_accuracy",
        "precision",
        "recall",
    ]
]

,period_end_date,rows,positive_rate,roc_auc,average_precision,balanced_accuracy,precision,recall
0,2019-06-30,304,0.240132,0.356995,0.195341,0.499615,0.238095,0.068493
1,2019-09-30,304,0.246711,0.304221,0.176724,0.456332,0.000000,0.000000
2,2019-12-31,305,0.242623,0.490055,0.347272,0.571048,0.517241,0.202703
3,2020-03-31,346,0.265896,0.758002,0.445031,0.712812,0.417989,0.858696
4,2020-06-30,346,0.265896,0.436195,0.225390,0.440389,0.223958,0.467391
5,2020-09-30,346,0.263006,0.817410,0.649820,0.633053,0.838710,0.285714
6,2020-12-31,346,0.265896,0.935253,0.837275,0.500000,0.000000,0.000000
7,2021-03-31,349,0.277937,0.830142,0.629477,0.505155,1.000000,0.010309
8,2021-06-30,360,0.252778,0.522284,0.245810,0.446097,0.000000,0.000000
9,2021-09-30,318,0.330189,0.272971,0.245606,0.495506,0.272727,0.028571


The quarterly out-of-fold results show that the logistic-regression baseline has two distinct issues: ranking instability and threshold/calibration instability. Some quarters have strong ROC-AUC but almost no positive predictions at the default 0.5 threshold. For example, 2020-12-31 achieved ROC-AUC 0.935 and Average Precision 0.837, yet recall was 0.000; 2021-03-31 also had strong ROC-AUC 0.830 but recall only 0.010. This means the model can sometimes rank future underperformers well, but its predicted probabilities are shifted too low for the default threshold to identify them. Therefore, the low recall in these quarters is not necessarily evidence of poor ranking.


Other quarters show a genuine ranking failure that cannot be fixed simply by lowering the classification threshold. The weakest examples are 2019-06-30 with ROC-AUC 0.357, 2019-09-30 with 0.304, and especially 2021-09-30 with 0.273. Fold 3 therefore was not uniformly poor: 2021-06-30 was approximately neutral (0.522), 2021-09-30 was a severe failure, while 2021-12-31 and 2022-03-31 recovered to 0.619 and 0.679. Conversely, Fold 4's strong pooled performance is supported by all four of its quarters: ROC-AUC ranged from about 0.631 to 0.880, suggesting the improvement was not caused by a single lucky quarter.


Because the target is defined as whether an investment option falls into the bottom quartile relative to peers in the same quarter, within-quarter ranking is particularly important. The current fold-level ROC-AUC pools four quarters together and can therefore compare observations that were never competing within the same peer group. The quarterly results suggest that this can obscure useful signal. For example, Fold 2 had pooled ROC-AUC around 0.647, while the mean of its four quarterly ROC-AUC values was roughly 0.755. Going forward, model evaluation should therefore report both pooled validation-block metrics and quarter-level metrics with macro-averaged performance, with the latter being especially relevant to this peer-relative target.

Audit missingness across the folds!

because it affects the model performance in a way that:<br>
Fold 4 strong because later periods contain substantially richer SAA/hedging information than earlier periods <br>
OR <BR>
Is saa_missing itself acting partly as a proxy for historical period?

Neither automatically constitutes leakage, but it could make the relationship learned in one historical regime unstable in another.

In [40]:
fold_missingness = []

for fold in folds:

    train_data = (
        development_data.loc[
            fold["train_index"]
        ]
    )

    validation_data = (
        development_data.loc[
            fold["validation_index"]
        ]
    )

    fold_missingness.append(
        {
            "fold":
                fold["fold"],

            "train_saa_missing_rate":
                train_data[
                    "saa_missing"
                ].mean(),

            "validation_saa_missing_rate":
                validation_data[
                    "saa_missing"
                ].mean(),

            "train_volatility_5y_missing_rate":
                train_data[
                    "volatility_5y_missing"
                ].mean(),

            "validation_volatility_5y_missing_rate":
                validation_data[
                    "volatility_5y_missing"
                ].mean(),

            "train_volatility_10y_missing_rate":
                train_data[
                    "volatility_10y_missing"
                ].mean(),

            "validation_volatility_10y_missing_rate":
                validation_data[
                    "volatility_10y_missing"
                ].mean(),
        }
    )

fold_missingness = pd.DataFrame(
    fold_missingness
)

fold_missingness

,fold,train_saa_missing_rate,validation_saa_missing_rate,train_volatility_5y_missing_rate,validation_volatility_5y_missing_rate,train_volatility_10y_missing_rate,validation_volatility_10y_missing_rate
0,1,0.420647,0.427323,0.699853,0.451152,0.788758,0.772041
1,2,0.439863,0.391492,0.667988,0.509012,0.773725,0.810382
2,3,0.436731,0.283892,0.613844,0.434609,0.773304,0.780702
3,4,0.426972,0.073485,0.591227,0.427273,0.781303,0.824242


The baseline's recent performance improvement coincides with a substantial change in SAA data availability. SAA missingness in validation falls from approximately 43% in Fold 1 to only 7% in Fold 4, while accumulated training missingness remains around 43%. Five-year volatility coverage improves more moderately, while ten-year volatility remains highly sparse throughout. This suggests that changing SAA coverage may contribute to temporal model instability and to the stronger recent validation results, although causality cannot yet be established.

Before adding a sophisticated model, let's determine where the signal comes from, which features.

In [43]:
feature_sets = {
    "performance_only":
        performance_features,

    "performance_volatility":
        (
            performance_features
            + volatility_features
        ),

    "performance_saa":
        (
            performance_features
            + saa_features
        ),

    "performance_saa_hedging":
        (
            performance_features
            + saa_features
            + hedging_features
        ),

    "all_baseline":
        baseline_features,

    "all_without_missing_flags": [
    feature
    for feature in baseline_features
    if feature not in [
        "saa_missing",
        "volatility_5y_missing",
        "volatility_10y_missing",
    ]
]
}

In [44]:
feature_sets

{'performance_only': ['return_measurement_comparison_percent',
  'return_lag_1q',
  'return_lag_2q',
  'return_lag_3q',
  'return_lag_4q',
  'return_change_1q',
  'return_trailing_4q',
  'return_mean_4q',
  'return_std_4q',
  'return_min_4q',
  'positive_quarter_share_4q',
  'current_return_minus_peer_median',
  'current_return_peer_percentile'],
 'performance_volatility': ['return_measurement_comparison_percent',
  'return_lag_1q',
  'return_lag_2q',
  'return_lag_3q',
  'return_lag_4q',
  'return_change_1q',
  'return_trailing_4q',
  'return_mean_4q',
  'return_std_4q',
  'return_min_4q',
  'positive_quarter_share_4q',
  'current_return_minus_peer_median',
  'current_return_peer_percentile',
  'return_investment_five_year_volatility_comparison_percent',
  'return_investment_ten_year_volatility_comparison_percent_clean',
  'volatility_5y_minus_10y',
  'volatility_5y_missing',
  'volatility_10y_missing'],
 'performance_saa': ['return_measurement_comparison_percent',
  'return_lag_1q',
